In [0]:
from pyspark.sql.functions import monotonically_increasing_id

silver_df = spark.table(
    "workforce.silver.employee"
)

dim_department = (

    silver_df

    .select("department")

    .distinct()

    .withColumn(
        "department_key",
        monotonically_increasing_id()
    )

    .select(
        "department_key",
        "department"
    )

)

display(dim_department)

In [0]:
(
    dim_department

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.dim_department"
    )
)

In [0]:
dim_job_role = (

    silver_df

    .select("job_role")

    .distinct()

    .withColumn(
        "job_role_key",
        monotonically_increasing_id()
    )

    .select(
        "job_role_key",
        "job_role"
    )

)
display(dim_job_role)

In [0]:
(
    dim_job_role

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.dim_job_role"
    )
)

In [0]:
dim_education = (

    silver_df

    .select(
        "education_field"
    )

    .distinct()

    .withColumn(
        "education_key",
        monotonically_increasing_id()
    )

)
display(dim_education)

In [0]:
(
    dim_education

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.dim_education"
    )
)

In [0]:
dim_employee = (

    silver_df

    .select(

        "employee_id",

        "age",

        "gender",

        "marital_status",

        "education",

        "education_field",

        "business_travel",

        "distance_from_home"

    )

)

dim_education=dim_education.drop("education")


In [0]:
(
    dim_employee

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.dim_employee"
    )
)

In [0]:
from pyspark.sql.functions import *

fct_workforce = (

    silver_df.alias("s")

    .join(
        dim_department.alias("d"),
        "department",
        "left"
    )

    .join(
        dim_job_role.alias("j"),
        "job_role",
        "left"
    )

    .join(
        dim_education.alias("e"),
        ["education_field"],
        "left"
    )

    .select(

        col("employee_id"),

        col("department_key"),

        col("job_role_key"),

        col("education_key"),

        col("monthly_income"),

        col("daily_rate"),

        col("hourly_rate"),

        col("monthly_rate"),

        col("years_at_company"),

        col("years_in_current_role"),

        col("years_since_last_promotion"),

        col("years_with_curr_manager"),

        col("total_working_years"),

        col("training_times_last_year"),

        col("job_satisfaction"),

        col("environment_satisfaction"),

        col("relationship_satisfaction"),

        col("work_life_balance"),

        col("job_involvement"),

        col("performance_rating"),

        col("stock_option_level"),

        col("over_time"),

        col("attrition"),

        col("num_companies_worked"),

        col("percent_salary_hike"),

        col("job_level")


    )

)
display(fct_workforce)

In [0]:
(
    fct_workforce

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.fct_workforce"
    )
)